# Predicciones — Blind Test Data
## ML Modeling Challenge - Wizeline

Generación de predicciones sobre `blind_test_data.csv` usando el modelo **CatBoost optimizado**
guardado en el Notebook 3.

- Modelo  : cargado desde `models/catboost_optimized.joblib`
- Features : leídas desde `config.yaml`
- Output  : `data/predictions.csv`

## 0. Imports y configuración.

In [1]:
from pathlib import Path

import joblib
import plotly.express as px
import polars as pl
import yaml

with open('config.yaml') as f:
    config = yaml.safe_load(f)

FEATURES: list[str] = config['features']['selected_v2']
MODEL_PATH: Path = Path(config['model']['artifact_path'])
BLIND_TEST_PATH: Path = Path(config['data']['blind_test_path'])
OUTPUT_PATH: Path = Path('data/predictions.csv')

print(f'Modelo          : {MODEL_PATH}')
print(f'Features ({len(FEATURES)})   : {FEATURES}')
print(f'Blind test path : {BLIND_TEST_PATH}')
print(f'Output path     : {OUTPUT_PATH}')


Modelo          : models/catboost_optimized.joblib
Features (5)   : ['feature_2', 'feature_13', 'feature_9', 'feature_18', 'feature_11']
Blind test path : data/blind_test_data.csv
Output path     : data/predictions.csv


## 1. Carga del modelo y datos.

In [2]:
pipeline = joblib.load(MODEL_PATH)
print(f'Modelo cargado: {type(pipeline.named_steps["model"]).__name__}')
print(f'Parámetros del estimador:')
for k, v in pipeline.named_steps['model'].get_params().items():
    print(f'  {k}: {v}')


Modelo cargado: CatBoostRegressor
Parámetros del estimador:
  iterations: 794
  learning_rate: 0.03327139059418807
  depth: 5
  l2_leaf_reg: 4.824999357282307
  loss_function: RMSE
  random_seed: 5000
  verbose: 0
  subsample: 0.6239194033296082


In [3]:
df_blind = pl.read_csv(BLIND_TEST_PATH)

## 2. Generación de predicciones.

In [4]:
X_blind = df_blind.select(FEATURES).to_numpy()
predictions = pipeline.predict(X_blind)

df_predictions = df_blind.with_columns(
    pl.Series('target_pred', predictions)
)

## 3. Pequeña evaluación de predicciones.

In [5]:
df_train_target = pl.read_csv('data/training_data.csv').select('target')

df_plot = pl.concat([
    df_train_target.rename({'target': 'valor'}).with_columns(pl.lit('Train (target)').alias('conjunto')),
    pl.DataFrame({'valor': predictions, 'conjunto': ['Blind Test (pred)'] * len(predictions)}),
])

fig = px.histogram(
    df_plot.to_pandas(),
    x='valor',
    color='conjunto',
    nbins=30,
    barmode='overlay',
    opacity=0.65,
    title='Distribución de target (Train) vs predicciones (Blind Test)',
    labels={'valor': 'target / target_pred', 'count': 'Frecuencia'},
    color_discrete_map={'Train (target)': '#EF553B', 'Blind Test (pred)': '#636EFA'},
)
fig.update_layout(height=450, legend_title_text='Conjunto')
fig.show()

__NOTAS:__
1. Comparando las distribuciones del target en el conjunto de entrenamiento y en el blind test, se observa que ambas caen en el mismo rango [0–30], lo que indica que el modelo no está extrapolando fuera del dominio conocido.
2. Las dos distribuciones tienen formas similares: aproximadamente simétricas y unimodales, con mayor concentración entre 10 y 20. Esto sugiere que el modelo aprendió correctamente la estructura global del target.
3. No se observan predicciones anómalas ni picos aislados, lo cual da confianza de que el pipeline de predicción funciona adecuadamente.

## 4. Guardado de predicciones.

In [6]:
df_output = pl.DataFrame({'target_pred': predictions})

df_output.write_csv(OUTPUT_PATH)

print(f'Predicciones guardadas en : {OUTPUT_PATH}')

Predicciones guardadas en : data/predictions.csv
